<a href="https://colab.research.google.com/github/F0nkell/Deepfake_Yandex/blob/main/Model_Architecture_Makeev_Ivan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
  def __init__(self):
    super(SimpleModel, self).__init__()
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(in_features=256*256*3, out_features=1)

  def forward(self, x):
    x = self.flatten(x)
    x = self.fc(x)
    return x


In [2]:
class DeepfakeCNN_v1(nn.Module):
  def __init__(self):
    super(DeepfakeCNN_v1, self).__init__()

    #Извлечение признаков
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels=16, kernel_size=3, padding=1)
    self.relu = nn.ReLU()
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    #Классификатор
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(in_features = 16 * 128 * 128, out_features = 1)

  def forward(self, x):
    #Пропускаем картинку через блок зрения
    x = self.conv1(x)
    x = self.relu(x)
    x = self.pool(x)
    #Классифицируем
    x = self.flatten(x)
    x = self.fc(x)
    return x

In [3]:
dummy_data = torch.randn(5, 3, 256, 256)

model = DeepfakeCNN_v1()

predictions = model(dummy_data)
print(f"Размерность входных данных: {dummy_data.shape}")
print(f"Размерность предсказаний: {predictions.shape}")
print(f"Сами предсказания (сырые логиты):\n {predictions}")

Размерность входных данных: torch.Size([5, 3, 256, 256])
Размерность предсказаний: torch.Size([5, 1])
Сами предсказания (сырые логиты):
 tensor([[0.4067],
        [0.0958],
        [0.4183],
        [0.6901],
        [0.6193]], grad_fn=<AddmmBackward0>)


In [4]:
class DeepfakeCNN_v2(nn.Module):
  def __init__(self):
    super(DeepfakeCNN_v2, self).__init__()

   #Блок 1: Извлечение низкоуровневых признаков (контуры и границы)
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels=32, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(num_features=32)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Блок 2: Выделение текстурных особенностей и паттернов
    self.conv2 = nn.Conv2d(in_channels = 32, out_channels=64, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(num_features=64)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Блок 3: Распознавание сложных структурных элементов лица
    self.conv3 = nn.Conv2d(in_channels = 64, out_channels=128, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(num_features=128)
    self.relu3 = nn.ReLU()
    self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)


    # Блок 4: Анализ высокоуровневых признаков и артефактов синтеза
    self.conv4 = nn.Conv2d(in_channels = 128, out_channels=256, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(num_features=256)
    self.relu4 = nn.ReLU()
    self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Классификатор
    self.flatten = nn.Flatten()
    self.relu_fc = nn.ReLU()
    self.fc2 = nn.Linear(512, 1)

  def forward(self, x):
    #Пропускаем картинку через блок зрения
    x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
    x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
    x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
    x = self.pool4(self.relu4(self.bn4(self.conv4(x))))

    x = self.flatten(x)
    x = self.relu_fc(self.fc1(x))
    x =  self.fc2(x)

    return x

In [5]:
dummy_data = torch.randn(8, 3, 256, 256)

model = DeepfakeCNN_v1()

predictions = model(dummy_data)
print(f"Размерность входных данных: {dummy_data.shape}")
print(f"Размерность предсказаний: {predictions.shape}")
print(f"Сами предсказания (сырые логиты):\n {predictions}")

Размерность входных данных: torch.Size([8, 3, 256, 256])
Размерность предсказаний: torch.Size([8, 1])
Сами предсказания (сырые логиты):
 tensor([[ 0.0590],
        [-0.2819],
        [-0.1909],
        [-0.1459],
        [-0.1819],
        [-0.1633],
        [ 0.2718],
        [-0.2743]], grad_fn=<AddmmBackward0>)


In [6]:
class DeepfakeCNN_Final(nn.Module):
  def __init__(self):
    super(DeepfakeCNN_Final, self).__init__()

   #Блок 1: Извлечение низкоуровневых признаков (контуры и границы)
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels=32, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(num_features=32)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Блок 2: Выделение текстурных особенностей и паттернов
    self.conv2 = nn.Conv2d(in_channels = 32, out_channels=64, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(num_features=64)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Блок 3: Распознавание сложных структурных элементов лица
    self.conv3 = nn.Conv2d(in_channels = 64, out_channels=128, kernel_size=3, padding=1)
    self.bn3 = nn.BatchNorm2d(num_features=128)
    self.relu3 = nn.ReLU()
    self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)


    # Блок 4: Анализ высокоуровневых признаков и артефактов синтеза
    self.conv4 = nn.Conv2d(in_channels = 128, out_channels=256, kernel_size=3, padding=1)
    self.bn4 = nn.BatchNorm2d(num_features=256)
    self.relu4 = nn.ReLU()
    self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

    #Классификатор
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(p=0.5)
    self.fc1 = nn.Linear(256 * 16 * 16, 512)
    self.relu_fc = nn.ReLU()
    self.fc2 = nn.Linear(512, 1)

  def forward(self, x):
    #Пропускаем картинку через блок зрения
    x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
    x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
    x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
    x = self.pool4(self.relu4(self.bn4(self.conv4(x))))

    x = self.flatten(x)
    x = self.dropout(x) # Применяем защиту перед первым плотным слоем
    x = self.relu_fc(self.fc1(x))
    x = self.fc2(x)

    return x